# ToolError Recovery Demo

This notebook demonstrates how the agent can automatically recover from errors using the new `ToolError` protocol migration. 

We will tell the agent to read a file that doesn't exist. The agent should:
1. Receive a `ToolError`.
2. List the directory to find the actual files.
3. Succeed in its task.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

from agents.mcp import MCPServerStdio
from agents import Agent, Runner, RunConfig, trace
from agents.extensions.models.litellm_model import LitellmModel


load_dotenv(Path("..") / ".env")

# Workspace setup
root_dir = Path(os.getcwd()).parent
abs_tmp_dir = str((root_dir / "tmp").resolve())
os.makedirs(abs_tmp_dir, exist_ok=True)

# Create a file with a slightly different name than expected
with open(os.path.join(abs_tmp_dir, "hidden_data.txt"), "w") as f:
    f.write("Secret Password: antigravity")

mcp_server = MCPServerStdio(
    name="Sandboxed Workspace",
    params={ 
        "command": "docker", 
        "args": [ 
            "run", "-i", "--rm", 
            "--user", "1000:1000", 
            "--security-opt", "no-new-privileges",
            "--cap-drop", "ALL",
            "--init",
            "--memory", "512m",
            "--cpus", "0.5",
            "-v", f"{abs_tmp_dir}:/workspace", 
            "agent-workspace-mcp" 
        ] 
    }, 
    client_session_timeout_seconds=60.0 
)

model_name = os.environ.get("DEFAULT_MODEL", "openrouter/google/gemini-2.0-flash-001")

agent = Agent(
    name="RecoveryExpert",
    instructions=(
        "You are a problem solver."
    ),
    model=LitellmModel(model=model_name),
    mcp_servers=[mcp_server]
)

async def run_demo():
    mission = """Tell me the password stored in 'secret.txt'. "
        "Do NOT list the directory, directly read the file. """
    print(f"🚀 Starting Mission: {mission}\n")
    
    async with mcp_server:
        # Enable tracing for observability
        with trace("Error-Recovery-Demo"):
            # Use run_streamed to see intermediary results
            stream = Runner.run_streamed(agent, mission, max_turns=10, run_config=RunConfig())
            async for event in stream.stream_events():
                if event.type == "run_item_stream_event":
                    item = event.item
                    if event.name == "tool_called":
                        print(f"\n🛠️  [TOOL CALL] {item.raw_item.name}({item.raw_item.arguments})")
                    elif event.name == "tool_output":
                        # Truncate large results for readability
                        output = str(item.output)
                        if len(output) > 200: 
                            output = output[:200] + "..."
                        print(f"✅ [RESULT] {output}")
                elif event.type == "raw_response_event":
                    from openai.types.responses import ResponseTextDeltaEvent
                    if isinstance(event.data, ResponseTextDeltaEvent):
                        print(event.data.delta, end="", flush=True)

await run_demo()